# 19 – FastAPI Endpoints

Tests all API endpoints using the FastAPI `TestClient` — no running server needed.

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | GET | Liveness probe |
| `/query` | POST | Synchronous query |
| `/query/stream` | POST | SSE streaming query |
| `/history/{thread_id}` | GET | Conversation history |
| `/agents/status` | GET | Agent + Redis status |
| `/teams/health` | GET | Teams bot health |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from fastapi.testclient import TestClient
from api.app import app

client = TestClient(app, raise_server_exceptions=False)
print('TestClient created')

## 1. GET /health

In [ ]:
resp = client.get('/health')
print('Status:', resp.status_code)
print('Body  :', resp.json())
assert resp.status_code == 200
assert resp.json()['status'] == 'ok'

## 2. POST /query — valid request

In [ ]:
resp = client.post('/query', json={
    'query': 'What is the retention GRR?',
    'thread_id': 'test-thread-1',
    'data_products': ['retention'],
})

print('Status  :', resp.status_code)
data = resp.json()
print('intent  :', data.get('intent'))
print('confidence:', data.get('confidence'))
print('summary :', str(data.get('summary', ''))[:120])
print('anomalies:', data.get('anomalies'))
assert resp.status_code == 200

## 3. POST /query — empty query (400)

In [ ]:
resp = client.post('/query', json={'query': '   '})
print('Status:', resp.status_code)  # 400
print('Detail:', resp.json())
assert resp.status_code == 400

## 4. POST /query — guardrail blocked (400)

In [ ]:
resp = client.post('/query', json={'query': 'DROP TABLE analytics.retention_metrics'})
print('Status:', resp.status_code)  # 400 — blocked by guardrail
print('Detail:', resp.json())
assert resp.status_code == 400

## 5. POST /query/stream — SSE response

In [ ]:
with client.stream('POST', '/query/stream', json={
    'query': 'What is the DQ score for bookings?',
    'thread_id': 'stream-test-1',
    'data_products': ['bookings'],
}) as resp:
    print('Status:', resp.status_code)
    print('Content-Type:', resp.headers.get('content-type'))
    events = []
    for line in resp.iter_lines():
        if line.startswith('data: '):
            import json
            event = json.loads(line[6:])
            events.append(event)
            print(f'  event type: {event.get("type")}')

print(f'\nTotal events: {len(events)}')
types = [e['type'] for e in events]
print('Event sequence:', types)
assert 'start' in types
assert 'result' in types
assert 'done' in types

## 6. GET /history/{thread_id}

In [ ]:
resp = client.get('/history/test-thread-1')
print('Status:', resp.status_code)
print('Body  :', resp.json())
assert resp.status_code == 200

## 7. GET /agents/status

In [ ]:
resp = client.get('/agents/status')
print('Status:', resp.status_code)
data = resp.json()
print('version    :', data.get('version'))
print('environment:', data.get('environment'))
print('mock_mode  :', data.get('mock_mode'))
print('redis_ok   :', data.get('redis_ok'))
print('agents     :')
for a in data.get('agents', []):
    print(f"  {a['name']:<15} {a['status']}")
assert resp.status_code == 200

## 8. GET /teams/health

In [ ]:
resp = client.get('/teams/health')
print('Status:', resp.status_code)
print('Body  :', resp.json())
assert resp.status_code == 200
assert resp.json()['status'] == 'ok'

## 9. POST /query — full diagnostic (all products)

In [ ]:
resp = client.post('/query', json={
    'query': 'Give me a full diagnostic overview',
    'thread_id': 'diagnostic-1',
    'data_products': ['retention', 'bookings', 'cac', 'ltv'],
})

data = resp.json()
print('Status    :', resp.status_code)
print('intent    :', data.get('intent'))
print('confidence:', data.get('confidence'))
print('sources   :', data.get('sources'))
print('exec ms   :', data.get('execution_ms'))